# measure acceleration

## author:
- **David W. Hogg** (NYU) (Flatiron) (MPIA)

## bugs:
- Needs to be audited for units (days vs seconds).

## project:
- See if I can measure an acceleration in one of our precise *Kepler* clocks.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pylab as plt

In [ ]:
# all this to import the KeplerClocks code

from pathlib import Path
import sys
target_dir = Path.cwd() / "../../KeplerClocks/py"
sys.path.append(str(target_dir.resolve()))
import clocks as kc

In [ ]:
seconds_per_day = 86_400
rng = np.random.default_rng(17)

In [ ]:
# read in some useful clocks from a previous life

CLOCKS_DB_FILE = Path.cwd() / "../../KeplerClocks/data/clocks_2026-09-14.db"
conn = kc.get_db_connection(CLOCKS_DB_FILE)
query = "SELECT * FROM best_clock ORDER BY theoretical_value DESC;"
clocks = pd.read_sql_query(query, conn)
conn.close()
print(clocks[:5])

In [ ]:
# pull one clock and get the pars

ii = 4
clock = clocks.iloc[ii]
print(clock)

In [ ]:
def get_ominusc_for_one_clock(clock, nchunk=32, nboot=31):

    # get data
    ts, ys, errs, df, dt = kc.get_kepler_data(clock["star_id"])
    ivars = errs ** -2

    # fit and get residuals; project onto derivative
    X, ms, pars = kc.fourier_wls_fit(clock["angular_frequency"], clock["fourier_series_degree"],
                                     ts, ys, ivars)
    resids = ys - X @ pars
    deriv_pars = kc.take_derivative_wrt_phase(pars, ms)
    derivs = clock['angular_frequency'] * X @ deriv_pars # dflux / dt

    # average o-c in chunks
    chunk_ts, chunk_advances = np.zeros(nchunk), np.zeros(nchunk)
    chunk_ivars, chunk_boot_vars = np.zeros(nchunk), np.zeros(nchunk)
    for chunk in range(nchunk):
        a, b = np.percentile(ts, [100 * chunk / nchunk, 100 * (chunk + 1) / nchunk])
        inchunk = (ts >= a) & (ts <= b)
        ninchunk = np.sum(inchunk)
        chunk_ts[chunk] = np.sum(ivars[inchunk] * ts[inchunk]) \
                        / np.sum(ivars[inchunk])
        chunk_advances[chunk] = np.sum(ivars[inchunk] * resids[inchunk] * derivs[inchunk]) \
                              / np.sum(ivars[inchunk] * derivs[inchunk] * derivs[inchunk])
        chunk_ivars[chunk]    = np.sum(ivars[inchunk] * derivs[inchunk] * derivs[inchunk])
        bootidx = inchunk.copy()
        boots = np.zeros(nboot)
        bootidx = np.arange(len(ts))[inchunk]
        for i in range(nboot):
            thisidx = rng.choice(bootidx, size=len(bootidx))
            boots[i] = np.sum(ivars[thisidx] * resids[thisidx] * derivs[thisidx]) \
                     / np.sum(ivars[thisidx] * derivs[thisidx] * derivs[thisidx])
        chunk_boot_vars[chunk] = np.var(boots)
    return chunk_ts, chunk_advances, chunk_ivars, chunk_boot_vars

In [ ]:
# is there a trend in the residuals?

def get_and_plot_one_ominusc(clock):
    chunk_ts, advances, _, boot_vars = get_ominusc_for_one_clock(clock)
    f = plt.figure(figsize=(9, 3))
    plt.axhline(0, c="k", lw=0.5)
    plt.step(chunk_ts, seconds_per_day * advances, c="k", where="mid")
    plt.errorbar(chunk_ts, seconds_per_day * advances,
                 yerr = seconds_per_day * np.sqrt(boot_vars),
                 marker=".", color="k", linestyle="none", label="bootstrap uncertainties")
    plt.legend()
    plt.xlabel("Barycentric time BJD [d]")
    plt.ylabel("advance $O-C$ [s]")
    plt.title(f"timing offsets of clock in {clock["star_id"]} with period {2. * np.pi / clock["angular_frequency"]:.5f} d")
    plt.show()
    return f

In [ ]:
for i, clock in clocks.iterrows():
    get_and_plot_one_ominusc(clock)